In [1]:
import matplotlib.pyplot as plt
import numpy as np
import heapq
import random
from autogen import AssistantAgent

# --- Initialize AI Agents with clear roles to skip "history block" prompt ---
llm_config = {
    "api_key": "---------API KEY-----------API KEY------------API KEY------------- ",
    "base_url": "https://api.groq.com/openai/v1",
    "model": "llama-3.3-70b-versatile"
}

planner = AssistantAgent(
    name="PlannerAgent",
    llm_config=llm_config,
    system_message="You are a route planner for a self-driving car. Generate the best route step-by-step using a grid map."
)
executor = AssistantAgent(
    name="ExecutorAgent",
    llm_config=llm_config,
    system_message="You are responsible for executing planned actions into actual commands for the self-driving car."
)
decision_maker = AssistantAgent(
    name="DecisionAgent",
    llm_config=llm_config,
    system_message="You make the final decision on whether the move is safe and valid. Only return 'Move to (x, y).' format."
)
perceptor = AssistantAgent(
    name="PerceptionAgent",
    llm_config=llm_config,
    system_message="You interpret the grid map and current environment for the car and detect obstacles, lights, etc."
)

# --- Car State ---
car = {'x': 2, 'y': 1, 'direction': 'E'}

# --- Movement Functions ---
def move_forward(car):
    if car['direction'] == 'N': car['x'] -= 1
    elif car['direction'] == 'S': car['x'] += 1
    elif car['direction'] == 'E': car['y'] += 1
    elif car['direction'] == 'W': car['y'] -= 1

def turn_left(car):
    directions = ['N', 'W', 'S', 'E']
    car['direction'] = directions[(directions.index(car['direction']) + 1) % 4]

def turn_right(car):
    directions = ['N', 'E', 'S', 'W']
    car['direction'] = directions[(directions.index(car['direction']) + 1) % 4]

# --- Obstacle Generator ---
def randomly_add_obstacle(grid, car, goal, chance=0.4):
    rows, cols = len(grid), len(grid[0])
    if random.random() < chance:
        while True:
            x, y = random.randint(0, rows - 1), random.randint(0, cols - 1)
            if grid[x][y] == 0 and (x, y) != (car['x'], car['y']) and (x, y) != goal:
                grid[x][y] = 1
                print(f"🚧 New obstacle added at: ({x}, {y})")
                break

# --- A* Pathfinding ---
def heuristic(a, b): return abs(a[0] - b[0]) + abs(a[1] - b[1])

def a_star_search(grid, start, goal):
    rows, cols = len(grid), len(grid[0])
    open_set = [(0, start)]
    came_from, g_score = {}, {start: 0}

    while open_set:
        _, current = heapq.heappop(open_set)
        if current == goal:
            path = []
            while current in came_from:
                path.append(current)
                current = came_from[current]
            path.reverse()
            return path

        for dx, dy in [(-1,0), (1,0), (0,-1), (0,1)]:
            nx, ny = current[0] + dx, current[1] + dy
            neighbor = (nx, ny)
            if 0 <= nx < rows and 0 <= ny < cols and grid[nx][ny] != 1:
                cost = 1 + (3 if grid[nx][ny] == 2 else 0)
                tentative = g_score[current] + cost
                if neighbor not in g_score or tentative < g_score[neighbor]:
                    g_score[neighbor] = tentative
                    heapq.heappush(open_set, (tentative + heuristic(neighbor, goal), neighbor))
                    came_from[neighbor] = current
    return None

# --- Grid Visualization ---
def draw_grid(grid, car, path=None):
    grid_np = np.array(grid)
    fig, ax = plt.subplots(figsize=(5, 5))
    ax.imshow(grid_np, cmap='gray_r')
    if path:
        for (x, y) in path:
            ax.plot(y, x, 'go', markersize=8)
    ax.plot(car['y'], car['x'], 'ro', markersize=12)
    plt.grid(True)
    plt.xticks(np.arange(len(grid[0])))
    plt.yticks(np.arange(len(grid)))
    plt.gca().invert_yaxis()
    plt.show()

# --- World Map ---
grid_map = [
    [0, 0, 0, 0, 0],
    [0, 1, 1, 2, 0],
    [0, 0, 0, 0, 0],
    [3, 1, 0, 1, 0],
    [0, 0, 0, 0, 0]
]

# --- Agent Pipeline ---
def multi_agent_decide_and_move(car, goal, grid_map):
    msg = f"""
You are a self-driving AI. The robot is at ({car['x']}, {car['y']}) facing {car['direction']}.
It needs to reach {goal}. The map is a 2D grid where:
0 = free, 1 = obstacle, 2 = red light, 3 = stop sign.
Avoid obstacles and red lights if possible. Plan next move step-by-step.
Only reply with the next move as: Move to (x, y).
    """

    try:
        perceptor_response = perceptor.initiate_chat(planner, msg)
        planner_response = planner.initiate_chat(executor, perceptor_response)
        executor_response = executor.initiate_chat(decision_maker, planner_response)
        decision_response = decision_maker.initiate_chat(planner, executor_response)

        print("\n--- Agent Responses ---")
        print("Perceptor:\n", perceptor_response)
        print("Planner:\n", planner_response)
        print("Executor:\n", executor_response)
        print("Decision:\n", decision_response)
        print("------------------------\n")

        if "move to" in decision_response.lower():
            try:
                coords = eval(decision_response.lower().split("move to")[1].split(".")[0].strip())
                if isinstance(coords, tuple) and len(coords) == 2:
                    print(f"✅ Car moved to: {coords}")
                    car['x'], car['y'] = coords
                    return True
            except Exception as e:
                print(" Error parsing move:", e)
        else:
            print(" No valid 'Move to' command found.")

    except Exception as err:
        print(" API call failed:", err)

    return False

# --- Test the API works before main loop ---
print(" Running test call to ensure API connectivity...")
test = planner.initiate_chat(executor, "You are in a simulation. Respond with 'Move to (2, 2).'")
print("Test Response:", test)

# --- Simulation Loop ---
goal = (4, 4)
for step in range(10):
    print(f"\n--- Step {step + 1} ---")
    randomly_add_obstacle(grid_map, car, goal)
    moved = multi_agent_decide_and_move(car, goal, grid_map)
    draw_grid(grid_map, car)

    if (car['x'], car['y']) == goal:
        print("Goal Reached!")
        break
    elif not moved:
        print(" Stuck. Ending simulation.")
        break


C:\Users\hankp\anaconda3\Lib\site-packages\paramiko\transport.py:219: CryptographyDeprecationWarning: Blowfish has been deprecated and will be removed in a future release
  "class": algorithms.Blowfish,


🌐 Running test call to ensure API connectivity...


> no history


PlannerAgent (to ExecutorAgent):

no history

--------------------------------------------------------------------------------
[autogen.oai.client: 05-02 08:56:28] {695} WARNING - Model llama-3.3-70b-versatile is not found. The cost will be 0. In your config_list, add field {"price" : [prompt_price_per_1k, completion_token_price_per_1k]} for customized pricing.
ExecutorAgent (to PlannerAgent):

I'm starting from a blank slate, with no prior knowledge or context. I'll respond based on the current conversation. 

We're discussing a self-driving car. What specific actions or commands would you like me to execute for the vehicle?

--------------------------------------------------------------------------------
[autogen.oai.client: 05-02 08:56:29] {695} WARNING - Model llama-3.3-70b-versatile is not found. The cost will be 0. In your config_list, add field {"price" : [prompt_price_per_1k, completion_token_price_per_1k]} for customized pricing.
PlannerAgent (to ExecutorAgent):

Let's assume 

RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01jqfxbsz2eh2s333nq0vjdrmw` service tier `on_demand` on tokens per day (TPD): Limit 100000, Used 94533, Requested 6089. Please try again in 8m57.157s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}